🔹 Tema A: Abstracción de Datos (ANSI/SPARC)
1. Tres niveles de abstracción

La arquitectura ANSI/SPARC define tres niveles:

Nivel Externo (Vista):
Representa cómo los usuarios ven los datos. Oculta la complejidad del sistema y muestra solo la información relevante para cada usuario.

Nivel Conceptual (Lógico):
Describe la estructura global de la base de datos, incluyendo tablas, relaciones y restricciones. Es independiente del almacenamiento físico.

Nivel Interno (Físico):
Define cómo se almacenan realmente los datos en el hardware (archivos, índices, bloques, etc.).

2. Independencia Lógica de Datos

Es la capacidad de modificar la estructura lógica sin afectar las aplicaciones.

Ejemplo:
Si se agrega una columna llamada edad en la tabla Usuarios, las aplicaciones que no utilizan esa columna seguirán funcionando sin cambios.

3. Independencia Física de Datos

Es la capacidad de cambiar el almacenamiento físico sin afectar la estructura lógica ni las aplicaciones.

Ejemplo:
Si se cambia un disco duro HDD por un SSD, la base de datos funcionará más rápido, pero las consultas SQL no necesitan modificarse.

🔹 Tema B: Impedance Mismatch y Modelos
4. Impedance Mismatch

Es el conflicto entre la Programación Orientada a Objetos (Python/Java) y el modelo relacional (SQL).

En programación, los datos se manejan como objetos complejos y jerárquicos, mientras que en SQL se almacenan en tablas planas. Esto obliga a transformar los datos constantemente, generando complejidad y pérdida de eficiencia.

5. Problema del Modelo Jerárquico

El modelo jerárquico falló porque solo permitía relaciones de tipo árbol (uno a muchos).

Ejemplo:
Un estudiante puede estar en varias clases y una clase tiene varios estudiantes (relación muchos a muchos).
El modelo jerárquico no podía representar esto sin duplicar datos.

6. Innovación de Edgar Codd

Edgar Codd introdujo el Álgebra Relacional.

Esto permitió separar:

Qué datos consultar (nivel lógico)

Cómo se obtienen (nivel físico)

7. JSON y Bases de Datos Documentales

Las bases de datos orientadas a documentos (como MongoDB) reducen el impedance mismatch porque almacenan los datos en formato JSON, similar a los objetos en programación.

Esto evita dividir la información en múltiples tablas y elimina la necesidad de JOINs complejos.

🔹 Tema C: Almacenamiento Físico
8. Costo en B-Tree

El principal costo es que al insertar en medio de la estructura se deben:

Reordenar datos

Mover registros en disco

Esto afecta el rendimiento.

9. Ventaja de Append-Only

Las arquitecturas Big Data usan append-only porque:

Solo agregan datos al final

No reordenan información

Son mucho más rápidas para escritura masiva

10. Lectura en Big Data

Se utilizan técnicas como:

Procesamiento distribuido (MapReduce, Spark)

Lectura en paralelo

Esto permite analizar grandes volúmenes de datos rápidamente.

🔹 Tema D: Escalamiento y Teorema CAP
11. Scale-Up vs Scale-Out

Escalabilidad Vertical (Scale-Up):
Consiste en mejorar un solo servidor (más RAM, CPU).
Es costosa y tiene límites físicos.

Escalabilidad Horizontal (Scale-Out):
Consiste en agregar más servidores.
Es más económica y permite crecer indefinidamente.

12. Límite del Scale-Up

El principal límite es el hardware:

Capacidad máxima de CPU

Memoria RAM

Calor y consumo energético

Por eso no se puede escalar indefinidamente un solo servidor.

13. Teorema CAP

Consistencia (C): Todos los nodos tienen la misma información al mismo tiempo.

Disponibilidad (A): El sistema siempre responde a las solicitudes.

Tolerancia a Particiones (P): El sistema sigue funcionando aunque haya fallas de red.

14. Por qué P no es opcional

En sistemas distribuidos siempre existe la posibilidad de fallas de red.

Por lo tanto, el sistema debe tolerar particiones, obligando a elegir entre:

Consistencia (CP)

Disponibilidad (AP)

15. Ejemplo de sistema AP

Ejemplo: Netflix o redes sociales

Estos sistemas priorizan la alta disponibilidad (AP).

Justificación:
No importa si el usuario ve información ligeramente desactualizada (por ejemplo, número de vistas o recomendaciones), pero sí es crítico que el sistema esté siempre disponible.

In [13]:
import sqlite3
import time
import random
import bisect

perfil_usuario = {
    "usuario_id": 9901,
    "username": "data_ninja",
    "metadatos": {"ip": "192.168.1.5", "ultima_conexion": "2026-03-09"},
    "compras": [
        {"item": "Laptop", "precio": 1500},
        {"item": "Mouse", "precio": 50}
    ],
    "amigos_ids": [45, 89, 102]
}

In [2]:
conn = sqlite3.connect("bootcamp.db")
cursor = conn.cursor()

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Usuarios (
    usuario_id INTEGER PRIMARY KEY,
    username TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Metadatos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER,
    ip TEXT,
    ultima_conexion TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Compras (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER,
    item TEXT,
    precio REAL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Amigos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER,
    amigo_id INTEGER
)
""")

In [4]:
cursor.execute("DELETE FROM Usuarios")
cursor.execute("DELETE FROM Metadatos")
cursor.execute("DELETE FROM Compras")
cursor.execute("DELETE FROM Amigos")
conn.commit()

In [5]:
# Usuario
cursor.execute("INSERT INTO Usuarios VALUES (?, ?)", 
               (perfil_usuario["usuario_id"], perfil_usuario["username"]))

# Metadatos
meta = perfil_usuario["metadatos"]
cursor.execute("INSERT INTO Metadatos (usuario_id, ip, ultima_conexion) VALUES (?, ?, ?)",
               (perfil_usuario["usuario_id"], meta["ip"], meta["ultima_conexion"]))

# Compras
for compra in perfil_usuario["compras"]:
    cursor.execute("INSERT INTO Compras (usuario_id, item, precio) VALUES (?, ?, ?)",
                   (perfil_usuario["usuario_id"], compra["item"], compra["precio"]))

# Amigos
for amigo in perfil_usuario["amigos_ids"]:
    cursor.execute("INSERT INTO Amigos (usuario_id, amigo_id) VALUES (?, ?)",
                   (perfil_usuario["usuario_id"], amigo))

conn.commit()

In [6]:
inicio = time.time()

cursor.execute("""
SELECT u.usuario_id, u.username, m.ip, m.ultima_conexion,
       c.item, c.precio, a.amigo_id
FROM Usuarios u
LEFT JOIN Metadatos m ON u.usuario_id = m.usuario_id
LEFT JOIN Compras c ON u.usuario_id = c.usuario_id
LEFT JOIN Amigos a ON u.usuario_id = a.usuario_id
WHERE u.usuario_id = 9901
""")

filas = cursor.fetchall()

resultado = {
    "usuario_id": filas[0][0],
    "username": filas[0][1],
    "metadatos": {
        "ip": filas[0][2],
        "ultima_conexion": filas[0][3]
    },
    "compras": [],
    "amigos_ids": []
}

compras_set = set()
amigos_set = set()

for fila in filas:
    if fila[4] is not None:
        compras_set.add((fila[4], fila[5]))
    if fila[6] is not None:
        amigos_set.add(fila[6])

resultado["compras"] = [{"item": i, "precio": p} for i, p in compras_set]
resultado["amigos_ids"] = list(amigos_set)

fin = time.time()

print("Resultado reconstruido:")
print(resultado)

print("\nTiempo:")
print(fin - inicio)

Resultado reconstruido:
{'usuario_id': 9901, 'username': 'data_ninja', 'metadatos': {'ip': '192.168.1.5', 'ultima_conexion': '2026-03-09'}, 'compras': [{'item': 'Laptop', 'precio': 1500.0}, {'item': 'Mouse', 'precio': 50.0}], 'amigos_ids': [89, 45, 102]}

Tiempo:
0.0009586811065673828


In [7]:
print("Filas planas:")
print(filas)

Filas planas:
[(9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Laptop', 1500.0, 45), (9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Laptop', 1500.0, 89), (9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Laptop', 1500.0, 102), (9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Mouse', 50.0, 45), (9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Mouse', 50.0, 89), (9901, 'data_ninja', '192.168.1.5', '2026-03-09', 'Mouse', 50.0, 102)]


In [8]:
b_tree_simulado = []
inicio = time.time()

for _ in range(100000):
    bisect.insort(b_tree_simulado, random.randint(1, 1000000))

tiempo_btree = time.time() - inicio
print("Tiempo B-Tree:", tiempo_btree)

Tiempo B-Tree: 1.808995008468628


In [9]:
data_lake = []
inicio = time.time()

for _ in range(100000):
    data_lake.append(random.randint(1, 1000000))

tiempo_append = time.time() - inicio

print("Tiempo Append:", tiempo_append)
print("Más rápido:", tiempo_btree / tiempo_append, "veces")

Tiempo Append: 0.0675044059753418
Más rápido: 26.798176834996855 veces


In [10]:
# B-Tree
inicio = time.time()
pos = bisect.bisect_left(b_tree_simulado, 999999)
t1 = time.time() - inicio

# Data Lake
inicio = time.time()
found = 999999 in data_lake
t2 = time.time() - inicio

print("Lectura B-Tree:", t1)
print("Lectura Data Lake:", t2)

Lectura B-Tree: 0.0
Lectura Data Lake: 0.0


In [11]:
class NodoDistribuido:
    def __init__(self, nombre):
        self.nombre = nombre
        self.inventario = {"iphone_15": 10}
        self.nodos_vecinos = []

    def conectar(self, nodo):
        self.nodos_vecinos.append(nodo)

    def comprar_item(self, item, cantidad):
        if self.inventario[item] >= cantidad:
            self.inventario[item] -= cantidad
            print(self.nombre, "stock:", self.inventario[item])
        else:
            print(self.nombre, "sin stock")

In [12]:
tokyo = NodoDistribuido("Tokyo")
ny = NodoDistribuido("New York")

# Romper red
tokyo.nodos_vecinos = []
ny.nodos_vecinos = []

tokyo.comprar_item("iphone_15", 8)
ny.comprar_item("iphone_15", 5)

print("\nInventarios finales:")
print("Tokyo:", tokyo.inventario)
print("NY:", ny.inventario)

Tokyo stock: 2
New York stock: 5

Inventarios finales:
Tokyo: {'iphone_15': 2}
NY: {'iphone_15': 5}
